# Notebook 10 — Anatomy of an API Call
**AI Models and Agents for Business · Week 5 · Session 10**

You have already called a model from code (Sessions 1 and 2). Today you look *underneath* that call, at what actually travels across the internet, because that is where the business questions live: what it costs, what limits apply, and what can break.

### What you will do
1. Send a **raw API request** and read every part of it.
2. Find the answer, and the **token counts**, inside the response.
3. Turn tokens into **dollars** and see what a product costs at scale.
4. Watch a conversation get **more expensive** with every turn.
5. Break the call on purpose: bad key, bad model, too many requests.
6. Decide what a product built on this API is **dependent** on (the Monitter lesson).

### Ground rules
Every cell is pre-written. You read, run, and change values in cells marked **✏️ CHANGE ME**. If something breaks, re-run from the top.

### One-time setup: store your key in Colab Secrets
1. Click the **🔑 key icon** in the left sidebar of Colab.
2. Click **Add new secret**. Name: `GROQ_API_KEY`. Value: your key (it starts with `gsk_`).
3. Turn on **Notebook access** for this notebook.

Colab will remember the key for all your notebooks, so you never paste it into a cell. If you skip this, the setup cell will simply ask you to paste the key each time (the box hides what you type).

In [ ]:
# Run this cell first. Do not modify.
!pip install groq --quiet

import time, json, textwrap
from groq import Groq
import groq

def _get_key():
    """Look for the key in Colab Secrets first; otherwise ask for it (hidden as you type)."""
    try:
        from google.colab import userdata
        return userdata.get("GROQ_API_KEY")
    except Exception:
        import getpass
        return getpass.getpass("Paste your Groq API key (starts with gsk_): ")

API_KEY = _get_key()
client = Groq(api_key=API_KEY)
MODEL = "openai/gpt-oss-20b"         # if Groq retires this name, change it here only

# Groq retired llama-3.1-8b-instant for free accounts in August 2026, so this notebook uses
# openai/gpt-oss-20b. It is a reasoning model: before it answers, it writes hidden "thinking"
# tokens. They count toward max_tokens and are billed as output tokens. The lines below keep
# that thinking short and add room for it, so every max_tokens in this notebook still means
# "room for the answer".
REASONING_EFFORT = "low"      # "low", "medium" or "high"
REASONING_HEADROOM = 300      # extra tokens reserved for the hidden thinking
_groq_create = client.chat.completions.create
def _create_with_reasoning(*args, **kwargs):
    if "gpt-oss" in str(kwargs.get("model", "")):
        kwargs.setdefault("reasoning_effort", REASONING_EFFORT)
        kwargs.setdefault("include_reasoning", False)
        if kwargs.get("max_tokens"):
            kwargs["max_tokens"] += REASONING_HEADROOM
    return _groq_create(*args, **kwargs)
client.chat.completions.create = _create_with_reasoning

def _retry_after(err, default=15):
    """Seconds the free tier asks us to wait (from the error's retry-after header), capped at 60."""
    try:
        return min(60.0, float(err.response.headers.get("retry-after", default)))
    except Exception:
        return float(default)

def ask(prompt, system=None, temperature=0.0, max_tokens=300, messages=None, model=None):
    """Send one request to the model and return the full response object.
    Waits and retries if the free tier says 'slow down'."""
    if messages is None:
        messages = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": prompt}]
    for attempt in range(6):
        try:
            return client.chat.completions.create(model=model or MODEL, messages=messages,
                                                  temperature=temperature, max_tokens=max_tokens)
        except groq.RateLimitError as err:
            wait = _retry_after(err)
            print(f"(rate limit hit - waiting {wait:.0f} seconds, then trying again)")
            time.sleep(wait)
    raise RuntimeError("Still rate limited. Wait a minute and re-run this cell.")

def say(prompt, **kw):
    """Same as ask(), but returns just the text of the answer."""
    return (ask(prompt, **kw).choices[0].message.content or "").strip()

print("✅ Setup complete. Model:", MODEL)

## Part A — The raw request

An API call is just a message sent to a web address. It has four parts: an **endpoint** (where), **headers** (who you are), a **body** (what you want), and a **method** (`POST` here, meaning "I am sending you something").

The `Groq` library you used before hides all of this. Here is the same call, written out by hand.

In [ ]:
import requests

ENDPOINT = "https://api.groq.com/openai/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {API_KEY}",     # your key: proves who you are (and who gets billed)
    "Content-Type": "application/json",
}

# ✏️ CHANGE ME — change the question and the max_tokens limit
body = {
    "model": MODEL,
    "messages": [
        {"role": "system", "content": "You are a concise business analyst."},
        {"role": "user",   "content": "Name two risks of building a product on a third-party AI API."},
    ],
    "temperature": 0.2,
    "max_tokens": 400,
    "reasoning_effort": "low",     # this model thinks before answering; keep the thinking short
    "include_reasoning": False,    # send back only the answer, not the thinking
}

# Show the request, with the key masked
shown = dict(headers); shown["Authorization"] = "Bearer " + API_KEY[:4] + "..." + API_KEY[-4:]
print("METHOD  : POST")
print("ENDPOINT:", ENDPOINT)
print("HEADERS :", json.dumps(shown, indent=2))
print("BODY    :", json.dumps(body, indent=2))

In [ ]:
r = requests.post(ENDPOINT, headers=headers, json=body, timeout=30)

print("STATUS CODE:", r.status_code, r.reason)
print("ROUND TRIP :", round(r.elapsed.total_seconds() * 1000), "ms")
print()
data = r.json()
print(json.dumps(data, indent=2)[:1500])

💬 **Discuss:** Which part of the request tells the provider *who to bill*? Which part controls *how much you pay*? (Hint: look at `max_tokens`.)

### Status codes in one line
`200` = worked · `401` = your key is wrong · `404` = what you asked for does not exist · `429` = you are going too fast · `500` = the provider has a problem.

## Part B — Reading the response

The response is **JSON**: labeled values, nested inside each other. To pull out a value you follow the path, one key at a time. Python writes the path with square brackets.

In [ ]:
answer      = data["choices"][0]["message"]["content"]
finish      = data["choices"][0]["finish_reason"]
prompt_tok  = data["usage"]["prompt_tokens"]
output_tok  = data["usage"]["completion_tokens"]
reason_tok  = (data["usage"].get("completion_tokens_details") or {}).get("reasoning_tokens", 0)

print("ANSWER      :", answer)
print()
print("FINISH      :", finish, "   (stop = finished naturally; length = hit your max_tokens limit)")
print("INPUT TOKENS:", prompt_tok)
print("OUTPUT TOKENS:", output_tok, f"  (of which hidden thinking: {reason_tok})")

### ✏️ CHANGE ME
Set `"max_tokens": 15` in the request above and re-run Part A, then this cell. What is the `finish_reason` now, and what happened to the answer? Why would a product team care? (This model spends some tokens thinking before it answers, and those count toward the limit and the bill. With 15 you may get no answer at all.)

## Part C — Tokens to dollars

Providers charge **per token**, with separate prices for input (your prompt) and output (the model's answer). A token is roughly ¾ of an English word.

⚠️ The prices below are **illustrative**. In class we will open the provider's pricing page and replace them with the current numbers. Prices change often.

In [ ]:
# ✏️ CHANGE ME — replace with the current numbers from the pricing page (dollars per 1 million tokens)
PRICE_IN_PER_M  = 0.075   # openai/gpt-oss-20b list price, Oct 2026 (check the live page)
PRICE_OUT_PER_M = 0.30

def call_cost(prompt_tokens, output_tokens):
    return prompt_tokens * PRICE_IN_PER_M / 1_000_000 + output_tokens * PRICE_OUT_PER_M / 1_000_000

one_call = call_cost(prompt_tok, output_tok)
print(f"This call cost about ${one_call:.7f}")
print(f"1,000 calls like it: ${one_call * 1_000:.4f}")
print(f"1,000,000 calls like it: ${one_call * 1_000_000:,.2f}")

### Scale it up
A retailer wants a support assistant. Fill in your best estimates, then run.

In [ ]:
# ✏️ CHANGE ME
conversations_per_month = 50_000
messages_per_conversation = 6
avg_prompt_tokens = 900      # grows with history (see Part D); use a measured number if you have one
avg_output_tokens = 150

calls = conversations_per_month * messages_per_conversation
monthly_model_cost = calls * call_cost(avg_prompt_tokens, avg_output_tokens)
print(f"{calls:,} calls per month  ->  ${monthly_model_cost:,.2f} per month in model fees")
print(f"Per conversation: ${monthly_model_cost / conversations_per_month:.4f}")
print()
print("Compare with the human cost per ticket you computed in Session 2 (Notebook 2, Exercise 1).")
print("What other costs are NOT in this number? (hosting, review of errors, escalations, engineering time...)")

## Part D — Why long conversations cost more

The API has **no memory**. Every turn, your code must resend the *whole* conversation so far. Watch the input tokens grow.

In [ ]:
import matplotlib.pyplot as plt

history = [{"role": "system", "content": "You are a helpful support assistant for an outdoor-gear retailer. Be brief."}]
questions = [
    "My tent poles snapped on the first trip. What are my options?",
    "I bought it 40 days ago. Does that matter?",
    "Can I get a replacement instead of a refund?",
    "How long would a replacement take?",
    "Can you summarize everything we decided?",
    "Thanks. Anything else I should know?",
]

input_tokens_by_turn = []
for q in questions:
    history.append({"role": "user", "content": q})
    resp = ask(None, messages=history, max_tokens=80)
    input_tokens_by_turn.append(resp.usage.prompt_tokens)
    history.append({"role": "assistant", "content": resp.choices[0].message.content})

for i, t in enumerate(input_tokens_by_turn, 1):
    print(f"Turn {i}: {t} input tokens")

plt.bar(range(1, len(input_tokens_by_turn) + 1), input_tokens_by_turn, color="#CC0000")
plt.xlabel("Turn"); plt.ylabel("Input tokens sent"); plt.title("Each turn resends the whole conversation")
plt.show()
print()
print(f"Total input tokens billed: {sum(input_tokens_by_turn)}")
print(f"The last turn sent {input_tokens_by_turn[-1] / input_tokens_by_turn[0]:.1f}x as many input tokens as the first.")

💬 **Discuss:** Your chart shows a growing staircase. If a conversation runs 40 turns, what happens to the cost of the *last* message compared with the first? What could a product team do about it? (Hint: summarize old turns, or start fresh.)

## Part E — Break it on purpose

Real systems fail. Provoke three failures so you recognize them.

In [ ]:
def raw_call(key=None, model=None, text="Say hello.", max_tokens=200):
    h = {"Authorization": f"Bearer {key or API_KEY}", "Content-Type": "application/json"}
    b = {"model": model or MODEL, "messages": [{"role": "user", "content": text}], "max_tokens": max_tokens}
    if "gpt-oss" in b["model"]:
        b["reasoning_effort"] = "low"     # only reasoning models accept this setting
    return requests.post(ENDPOINT, headers=h, json=b, timeout=30)

# 1. A wrong key
r1 = raw_call(key="gsk_this_is_not_a_real_key")
print("Wrong key   ->", r1.status_code, r1.json().get("error", {}).get("message"))

# 2. A model that does not exist  (✏️ CHANGE ME: try a typo)
r2 = raw_call(model="llama-9-giant")
print("Wrong model ->", r2.status_code, r2.json().get("error", {}).get("message"))

# 3. A normal call, for comparison
r3 = raw_call()
print("Normal      ->", r3.status_code, "OK" if r3.status_code == 200 else r3.text[:100])

💬 **Discuss:** A model provider **retires** a model name your product uses. Which of these three errors would your customers see? How would your team find out?

## Part F — Rate limits and retrying

Providers cap how fast you can call (requests and tokens per minute). Go over and you get `429`. The response headers tell you where you stand.

In [ ]:
interesting = {k: v for k, v in r3.headers.items() if "ratelimit" in k.lower() or k.lower() == "retry-after"}
print("Rate-limit headers on the last call:")
for k, v in interesting.items():
    print(f"  {k}: {v}")
if not interesting:
    print("  (none shown on this call)")

In [ ]:
def call_with_retry(max_tries=4, **kw):
    """Retry on 429 with a growing wait ('exponential backoff'). Returns the final response."""
    wait = 1
    for attempt in range(1, max_tries + 1):
        r = raw_call(**kw)
        if r.status_code != 429:
            return r
        pause = float(r.headers.get("retry-after", wait))
        print(f"  429 on try {attempt}. Waiting {pause} s...")
        time.sleep(pause)
        wait *= 2
    return r

# ✏️ CHANGE ME — fire this many quick requests (try 10, then 40) and count the status codes
N_QUICK = 10
codes = []
for _ in range(N_QUICK):
    codes.append(call_with_retry().status_code)
print("Status codes:", {c: codes.count(c) for c in sorted(set(codes))})

## Part G — The Monitter test

In the case, a developer's whole business depended on someone else's API, and the provider changed the rules. Use what you just saw to fill in this table for **a product that depends on a model API** (for example, your Project 1 prototype). Double-click this cell to edit.

| What could change | How we would find out | What it would do to our product | What we could do about it |
|---|---|---|---|
| Price per token rises | | | |
| Model is retired or renamed | | | |
| Rate limits tighten | | | |
| Provider changes its terms of use | | | |
| Provider has an outage | | | |

---
## ✅ Notebook 10 complete — Reflection (submit with the notebook)
1. Name the four parts of an API request and say what each does.
2. Why does a longer conversation cost more, even if the latest message is short?
3. What is the difference between a `401` and a `429`, and who has to act in each case?
4. Pick one row from the Monitter table. What would you do *before* it happened?

*Next session: Tool Calling and MCP. Today you saw the request; next you will let the model ask your code to do things.*